# Lección 12.3 — Multi-cabeza, codificación posicional y el bloque Transformer

> **Módulo 12 · Atención y Transformers** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 110–130 min (lección con ejercicios intercalados ≈ 80 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 12.2 (auto-atención), Lección 9.8 (normalización), Lección 10.4 (conexiones residuales)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar por qué un Transformer usa **varias cabezas** de atención en paralelo.
- Demostrar que la auto-atención **no sabe el orden** de las palabras y cómo lo resuelve la **codificación posicional**.
- Dibujar un **bloque Transformer** y explicar para qué sirve cada pieza (atención, MLP, residuales, normalización).
- **Entrenar tu propio mini-Transformer** en PyTorch (en CPU, en segundos) y mirar qué aprendió cada cabeza.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.set_printoptions(precision=2, suppress=True)
plt.rcParams["figure.figsize"] = (6, 4)
device = "cuda" if torch.cuda.is_available() else "cpu"

def softmax(puntajes, eje=-1):
    e = np.exp(puntajes - np.max(puntajes, axis=eje, keepdims=True))
    return e / e.sum(axis=eje, keepdims=True)

def auto_atencion(X, W_Q, W_K, W_V):
    """La fórmula de la Lección 12.2: softmax(Q·Kᵀ/√d)·V. Devuelve (salida, pesos)."""
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    pesos = softmax(Q @ K.T / np.sqrt(Q.shape[1]))
    return pesos @ V, pesos

def mapa_calor(M, filas, columnas, titulo="", ax=None, numeros=True):
    ax = ax or plt.gca()
    ax.imshow(M, cmap="Blues")
    ax.set_xticks(range(len(columnas)), columnas)
    ax.set_yticks(range(len(filas)), filas)
    if numeros:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                claro = M[i, j] < M.min() + 0.6 * (M.max() - M.min())
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7,
                        color="black" if claro else "white")
    ax.set_title(titulo, fontsize=10)

print("✅ Todo listo · dispositivo:", device)

---
## 1. 📖 La intuición: un comité de lectores

Imagina que una editorial le da el mismo manuscrito a **varios lectores**, y cada uno lo lee buscando algo distinto:

- 📝 La **correctora** mira la palabra anterior para revisar la concordancia ("las casa" ❌).
- 🕵️ El **detective** busca a quién se refiere cada pronombre ("ella" → "María").
- 🎭 El **crítico** busca el tono: ¿qué palabras indican ironía o tristeza?

Al final juntan sus notas. Ninguno solo entiende todo, pero el comité sí.

Eso es la **atención multi-cabeza**: en lugar de una sola atención, el Transformer hace **varias en paralelo** (las **cabezas**). Cada una tiene sus propias matrices `W_Q`, `W_K`, `W_V`, así que **cada una puede buscar algo distinto**. Al final se concatenan sus resultados.

Recuerda que una cabeza reparte pesos que suman 1: si una sola cabeza quisiera mirar a la vez "la palabra anterior" y "el sujeto", tendría que **promediarlos** y perdería detalle. Con varias cabezas, cada búsqueda tiene su propio "presupuesto" de atención.

### ✋ Ejercicio 1 — Diseña tu comité
En la frase *"Ayer **los** niños **que** vivían junto al río **encontraron** un perro"*, describe qué buscaría una cabeza que ayude a entender cada palabra en negrita:
1. "los" → ¿qué buscaría?
2. "que" → ¿qué buscaría?
3. "encontraron" → ¿qué buscaría? (Pista: ¿quién encontró? ¿qué encontró?)

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

Respuestas razonables (hay más):
1. "los" → el **sustantivo que viene** ("niños") para concordar en género y número. ¡Necesita mirar hacia adelante!
2. "que" → el **sustantivo al que se refiere** ("niños").
3. "encontraron" → su **sujeto** ("niños", que está lejos: hay 5 palabras en medio) y su **objeto** ("perro").

"encontraron" necesita **dos búsquedas distintas a la vez** (sujeto y objeto): un caso perfecto para dos cabezas. En la Sección 5 verás que, al entrenar, las cabezas se especializan solas.
</details>

---
## 2. 📖 Cómo funciona la multi-cabeza

Con un vector de tamaño `d_model` y `h` cabezas:
1. Cada cabeza proyecta las palabras a vectores **más pequeños**, de tamaño `d_model / h`.
2. Cada cabeza hace su propia auto-atención (Lección 12.2).
3. Se **concatenan** los resultados (vuelven a medir `d_model`) y una última matriz `W_O` los mezcla.

```
            ┌─ cabeza 1: atención con vectores de 2 ─┐
X (5 × 4) ──┤                                        ├─ concatenar (5 × 4) ── · W_O ──► salida (5 × 4)
            └─ cabeza 2: atención con vectores de 2 ─┘
```

Lo hacemos a mano con 5 palabras, `d_model = 4` y **2 cabezas** (matrices al azar, como antes de entrenar):

In [ ]:
rng = np.random.default_rng(0)
palabras = ["el", "perro", "mordió", "al", "hombre"]
X = rng.normal(size=(5, 4))                      # 5 palabras, d_model = 4
d_model, h = 4, 2
d_cabeza = d_model // h                          # cada cabeza trabaja con vectores de 2

salidas, mapas = [], []
for c in range(h):
    W_Q, W_K, W_V = (rng.normal(size=(d_model, d_cabeza)) for _ in range(3))   # matrices PROPIAS de la cabeza
    salida_c, pesos_c = auto_atencion(X, W_Q, W_K, W_V)
    salidas.append(salida_c); mapas.append(pesos_c)

concatenado = np.concatenate(salidas, axis=1)
W_O = rng.normal(size=(d_model, d_model))
salida_final = concatenado @ W_O

print("Salida de cada cabeza:", [s.shape for s in salidas])
print("Concatenado          :", concatenado.shape)
print("Salida final         :", salida_final.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for c in range(h):
    mapa_calor(mapas[c], palabras, palabras, f"Cabeza {c}", axes[c])
plt.tight_layout(); plt.show()

### 🔮 Ejercicio 2 — ¿Más cabezas = modelo más grande?
GPT-2 usa `d_model = 768`. Compara una atención con **1 cabeza** contra una con **12 cabezas** (de tamaño 768 / 12 = 64 cada una). ¿Cuál tiene **más parámetros**? ¿12 veces más?

✍️ *Mi predicción:*

In [ ]:
for cabezas in [1, 12]:          # ✏️ prueba también 4 o 768
    capa = nn.MultiheadAttention(embed_dim=768, num_heads=cabezas)
    print(f"{cabezas:>3} cabezas → {sum(p.numel() for p in capa.parameters()):,} parámetros")

<details><summary><b>👀 Ver respuesta</b></summary>

**Exactamente los mismos** (2,362,368). Con más cabezas, cada una trabaja con vectores **más pequeños** (64 en vez de 768), así que el total de números en `W_Q`, `W_K`, `W_V` y `W_O` no cambia. La multi-cabeza no hace el modelo más grande: **reparte** el mismo presupuesto en varias búsquedas independientes.
</details>

---
## 3. 📖 ¿Qué pasa si…? El orden de las palabras no existe

Compara:
- *"El **perro** mordió al **hombre**."*
- *"El **hombre** mordió al **perro**."*

Mismas palabras, significado opuesto. Ahora mira la fórmula de la auto-atención: compara **cada palabra con cada palabra**, sin importar dónde está. No hay nada en ella que diga "esta palabra va primero".

### 🔮 Ejercicio 3 — El perro y el hombre
Pasamos las dos frases por la **misma** auto-atención. ¿Será distinto el vector de salida de **"perro"** en una frase y en la otra?

✍️ *Mi predicción:*

In [ ]:
W_Q, W_K, W_V = (rng.normal(size=(4, 4)) for _ in range(3))
vector = dict(zip(palabras, X))                                  # cada palabra conserva su vector

frase_1 = ["el", "perro", "mordió", "al", "hombre"]
frase_2 = ["el", "hombre", "mordió", "al", "perro"]
sal_1, _ = auto_atencion(np.array([vector[w] for w in frase_1]), W_Q, W_K, W_V)
sal_2, _ = auto_atencion(np.array([vector[w] for w in frase_2]), W_Q, W_K, W_V)

print("'perro' en la frase 1:", sal_1[frase_1.index("perro")])
print("'perro' en la frase 2:", sal_2[frase_2.index("perro")])

<details><summary><b>👀 Ver respuesta</b></summary>

**Idénticos.** Para la auto-atención, las dos frases son la misma **bolsa de palabras**: cada palabra mira al mismo conjunto de compañeras y obtiene el mismo resultado, solo que en otro lugar de la lista. El modelo no podría saber quién mordió a quién.

Hace falta **darle al modelo la posición** de cada palabra.
</details>

### 📖 La solución: sumar un "código de posición"
Antes de la primera capa, a cada palabra se le **suma** un vector que depende solo de su **posición** (1.ª, 2.ª, 3.ª…). Así, "perro" en la posición 2 y "perro" en la posición 5 empiezan como vectores **distintos**.

Hay dos formas comunes:

| Tipo | Cómo se obtiene el vector de cada posición | Quién lo usa |
|---|---|---|
| **Sinusoidal** (fija) | Una fórmula con ondas seno y coseno de distintas frecuencias | El Transformer original (2017) |
| **Aprendida** | Una tabla de vectores que se entrena como cualquier otro parámetro | GPT-2, BERT |

La sinusoidal tiene una analogía bonita: un **reloj**. La manecilla de los segundos gira rápido, la de los minutos más lento y la de las horas muy lento. Con las tres juntas sabes la hora exacta. Cada par de casillas del vector es una "manecilla" que gira a distinta velocidad según la posición.

In [ ]:
def codificacion_sinusoidal(n_posiciones, d):
    pos = np.arange(n_posiciones)[:, None]
    frecuencias = 1 / 10000 ** (np.arange(0, d, 2) / d)    # de manecilla rápida a muy lenta
    pe = np.zeros((n_posiciones, d))
    pe[:, 0::2] = np.sin(pos * frecuencias)
    pe[:, 1::2] = np.cos(pos * frecuencias)
    return pe

PE = codificacion_sinusoidal(50, 32)
plt.figure(figsize=(9, 4))
plt.imshow(PE.T, cmap="RdBu", aspect="auto")
plt.xlabel("posición en la frase"); plt.ylabel("casilla del vector")
plt.title("Codificación posicional sinusoidal: cada columna es el 'código' de una posición")
plt.colorbar(); plt.show()

### 🔮 Ejercicio 4 — ¿Se parecen las posiciones vecinas?
Medimos el **producto punto** (el parecido) entre el código de la posición **20** y el de todas las demás. ¿Qué forma esperas: un pico en 20 que baja suavemente hacia los lados, o valores al azar?

✍️ *Mi predicción:*

In [ ]:
parecido = PE @ PE[20]
plt.plot(parecido, "o-")
plt.axvline(20, color="gray", ls="--")
plt.xlabel("posición"); plt.ylabel("parecido con la posición 20"); plt.show()
print("Parecido con la 19, 20 y 21:", parecido[19:22])
print("Parecido con la 0 y la 49  :", parecido[[0, 49]])

<details><summary><b>👀 Ver respuesta</b></summary>

Un **pico en 20** (parecido 16, el máximo) que baja suavemente: las posiciones 19 y 21 se parecen mucho (≈15.3) y las lejanas bastante menos (≈10 con la posición 0, ≈6 con la 49). Gracias a esto, la atención puede aprender reglas como "mira a la palabra **de al lado**" con solo comparar códigos de posición. En la Sección 5 verás a un modelo aprender justo eso.
</details>

---
## 4. 📖 El bloque Transformer: reunión + reflexión

Un Transformer es una pila de **bloques idénticos**. Cada bloque tiene dos partes:

```
       x ─────────────────────┐
       │                      │  (atajo residual)
   Normalizar                 │
       │                      │
  Atención multi-cabeza       │   ← 🗣️ REUNIÓN: las palabras intercambian información
       │                      │
       + ◄────────────────────┘
       │ ─────────────────────┐
   Normalizar                 │
       │                      │
   MLP (2 capas densas)       │   ← 🧠 REFLEXIÓN: cada palabra procesa por su cuenta lo que escuchó
       │                      │
       + ◄────────────────────┘
       │
     salida (misma forma que x)
```

| Pieza | Para qué sirve | Ya la conoces de… |
|---|---|---|
| **Atención multi-cabeza** | Mezclar información **entre** palabras | Lecciones 12.1–12.3 |
| **MLP** (*feed-forward*) | Transformar cada palabra **por separado** (aquí vive gran parte del "conocimiento") | Lección 9.2 |
| **Atajo residual** (`x + …`) | Cada parte solo **suma** una corrección; facilita entrenar redes profundas | Lección 10.4 (ResNet) |
| **Normalización** (*LayerNorm*) | Mantener los números en un rango estable | Lección 9.8 |

Como la salida tiene la **misma forma** que la entrada, puedes apilar tantos bloques como quieras: GPT-2 pequeño apila 12; GPT-3, 96.

Así se ve un bloque en PyTorch (usa la atención multi-cabeza ya programada, `nn.MultiheadAttention`):

In [ ]:
class BloqueTransformer(nn.Module):
    def __init__(self, d_model, n_cabezas):
        super().__init__()
        self.norma1 = nn.LayerNorm(d_model)
        self.atencion = nn.MultiheadAttention(d_model, n_cabezas, batch_first=True)
        self.norma2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                                 nn.Linear(4 * d_model, d_model))

    def forward(self, x):
        h = self.norma1(x)
        a, pesos = self.atencion(h, h, h, average_attn_weights=False)   # 🗣️ reunión
        x = x + a                                                          # atajo residual
        x = x + self.mlp(self.norma2(x))                                   # 🧠 reflexión + atajo
        return x, pesos            # devolvemos también los pesos para poder mirarlos

bloque = BloqueTransformer(d_model=32, n_cabezas=4)
entrada = torch.randn(1, 8, 32)            # 1 frase, 8 tokens, vectores de 32
salida, pesos = bloque(entrada)
print("Entrada:", tuple(entrada.shape), "→ salida:", tuple(salida.shape))
print("Pesos de atención (frase, cabezas, tokens, tokens):", tuple(pesos.shape))

### 🔮 Ejercicio 5 — ¿Dónde están los parámetros?
En este bloque (`d_model = 32`), ¿qué parte crees que tiene **más parámetros**: la atención o el MLP? (Pista: el MLP agranda los vectores a 4 × 32 = 128 y los vuelve a achicar.)

✍️ *Mi predicción:*

In [ ]:
for nombre, parte in [("atención", bloque.atencion), ("MLP", bloque.mlp),
                      ("normalizaciones", nn.ModuleList([bloque.norma1, bloque.norma2]))]:
    print(f"{nombre:<16} {sum(p.numel() for p in parte.parameters()):>6,} parámetros")

<details><summary><b>👀 Ver respuesta</b></summary>

El **MLP** (8,352) tiene casi el **doble** que la atención (4,224). Las normalizaciones son casi nada (128). En los Transformers grandes pasa lo mismo: alrededor de **2/3** de los parámetros de cada bloque están en el MLP. La atención decide **qué información se mueve**; el MLP hace gran parte del **cálculo** con ella.
</details>

---
## 5. 📖 Tu primer Transformer: aprender a invertir secuencias

Vamos a entrenar un mini-Transformer para una tarea pequeña pero que **exige atención y posiciones**: recibir 8 dígitos y devolverlos **al revés**.

```
entrada:  3 1 4 1 5 9 2 6
salida :  6 2 9 5 1 4 1 3
```

Para escribir el dígito de la posición 0 de la salida, el modelo tiene que mirar la posición 7 de la entrada; para la 1, la 6… Es decir, tiene que **saber dónde está cada dígito**.

El modelo completo: *embedding* de dígitos + codificación posicional **aprendida** (como GPT-2) → 2 bloques Transformer → una capa final que da una probabilidad para cada dígito (0–9) en cada posición.

In [ ]:
LARGO, VOCAB = 8, 10      # secuencias de 8 dígitos (0–9)

class MiniTransformer(nn.Module):
    def __init__(self, d_model=32, n_cabezas=4, n_capas=2, con_posicion=True):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB, d_model)
        self.posicion = nn.Embedding(LARGO, d_model) if con_posicion else None
        self.bloques = nn.ModuleList(BloqueTransformer(d_model, n_cabezas) for _ in range(n_capas))
        self.norma = nn.LayerNorm(d_model)
        self.salida = nn.Linear(d_model, VOCAB)

    def forward(self, x):
        z = self.embedding(x)
        if self.posicion is not None:
            z = z + self.posicion(torch.arange(x.shape[1], device=x.device))   # sumar el código de posición
        todos_los_pesos = []
        for bloque in self.bloques:
            z, pesos = bloque(z)
            todos_los_pesos.append(pesos)
        return self.salida(self.norma(z)), todos_los_pesos

def lote(n, tarea="invertir", generador=None):
    x = torch.randint(0, VOCAB, (n, LARGO), generator=generador)
    y = x.flip(1) if tarea == "invertir" else x.sort(dim=1).values
    return x.to(device), y.to(device)

def entrenar(tarea="invertir", pasos=300, semilla=0, **config):
    torch.manual_seed(semilla)
    modelo = MiniTransformer(**config).to(device)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=3e-3)
    for paso in range(pasos):
        x, y = lote(128, tarea)
        logits, _ = modelo(x)
        perdida = nn.functional.cross_entropy(logits.reshape(-1, VOCAB), y.reshape(-1))
        optimizador.zero_grad(); perdida.backward(); optimizador.step()
    return modelo

@torch.no_grad()
def evaluar(modelo, tarea="invertir"):
    x, y = lote(2000, tarea, torch.Generator().manual_seed(123))   # 2,000 secuencias nuevas, siempre las mismas
    pred = modelo(x)[0].argmax(-1)
    return (pred == y).float().mean().item(), (pred == y).all(dim=1).float().mean().item()

print("✅ Modelo definido")

### 🔮 Ejercicio 6 — ¿Cuánto aprenderá en 300 pasos?
Un modelo que adivina al azar acierta 1 de cada 10 dígitos (10 %). Vamos a entrenar **300 pasos** (unos segundos en CPU).
1. ¿Qué porcentaje de **dígitos** crees que acertará?
2. ¿Y de **secuencias completas** (los 8 dígitos bien)?

✍️ *Mis predicciones:*

In [ ]:
inicio = time.time()
modelo = entrenar("invertir", pasos=300)
acierto_digitos, acierto_secuencias = evaluar(modelo)
print(f"Entrenado en {time.time() - inicio:.0f} s · parámetros: {sum(p.numel() for p in modelo.parameters()):,}")
print(f"Dígitos correctos   : {acierto_digitos:.1%}")
print(f"Secuencias perfectas: {acierto_secuencias:.1%}")

x_ej = torch.tensor([[3, 1, 4, 1, 5, 9, 2, 6]], device=device)
print("\nEntrada :", x_ej[0].tolist())
print("Predicho:", modelo(x_ej)[0].argmax(-1)[0].tolist())

<details><summary><b>👀 Ver respuesta</b></summary>

**≈100 %** de dígitos y de secuencias completas. Con solo ~26 mil parámetros y unos segundos de entrenamiento, el Transformer aprendió a invertir. La tarea es fácil **para un modelo con atención**: cada posición de salida solo tiene que "mirar" a la posición espejo de la entrada.
</details>

### 🔮 Ejercicio 7 — ¿Qué aprendió cada cabeza?
Miramos los pesos de atención de las **4 cabezas de la primera capa** para la secuencia de ejemplo. Antes de ejecutar, dibuja en papel cómo sería el mapa ideal para invertir (fila = posición de salida, columna = posición a la que mira).

✍️ *Mi dibujo:*

In [ ]:
_, pesos_capas = modelo(x_ej)
CAPA = 0          # ✏️ prueba también la capa 1
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for c in range(4):
    mapa_calor(pesos_capas[CAPA][0, c].detach().cpu().numpy(), range(LARGO), range(LARGO),
               f"Capa {CAPA} · cabeza {c}", axes[c], numeros=False)
plt.tight_layout(); plt.show()

# ¿Cuánto peso pone cada cabeza en la posición "espejo" (la que hay que copiar)?
x_eval, _ = lote(500, "invertir", torch.Generator().manual_seed(1))
with torch.no_grad():
    _, pesos_eval = modelo(x_eval)
espejo = torch.arange(LARGO - 1, -1, -1)
for capa in range(2):
    for c in range(4):
        w = pesos_eval[capa][:, c, torch.arange(LARGO), espejo].mean().item()
        print(f"capa {capa} · cabeza {c}: peso promedio en la posición espejo = {w:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

El patrón ideal es la **diagonal invertida** (de arriba a la derecha a abajo a la izquierda): la salida 0 mira a la entrada 7, la 1 a la 6…

Los números lo confirman:
- En la **capa 0**, las cuatro cabezas ya miran bastante al espejo (≈48–68 %), pero de forma **borrosa**: un borrador.
- En la **capa 1** el patrón se vuelve casi perfecto (dos cabezas con ≈97 %, las otras ≈80 %): la segunda capa **afina** el trabajo de la primera.

Nadie le dijo al modelo "mira al espejo": **lo descubrió solo** porque así bajaba la pérdida. Como la tarea es tan simple, aquí las cabezas hacen casi lo mismo (son **redundantes**). En una tarea más rica, como el lenguaje del Proyecto del módulo, verás cabezas con trabajos muy distintos. ✏️ Con otra semilla el reparto cambia, pero la diagonal invertida aparece siempre.
</details>

### 🔮 Ejercicio 8 — Sin codificación posicional
Entrenamos el **mismo** modelo pero **sin** sumar el código de posición (`con_posicion=False`). Con lo que viste en la Sección 3, ¿qué exactitud esperas? ¿10 % (azar), 100 %, o algo intermedio?

✍️ *Mi predicción:*

In [ ]:
modelo_sin_pos = entrenar("invertir", pasos=300, con_posicion=False)
d_sin, s_sin = evaluar(modelo_sin_pos)
print(f"SIN posición → dígitos: {d_sin:.1%} · secuencias perfectas: {s_sin:.1%}")
print("Entrada :", x_ej[0].tolist())
print("Predicho:", modelo_sin_pos(x_ej)[0].argmax(-1)[0].tolist())

<details><summary><b>👀 Ver respuesta</b></summary>

Algo intermedio, pero muy malo: ≈**32 %** de dígitos y **0 %** de secuencias. Sin posiciones, todas las salidas "ven" la misma bolsa de dígitos, así que el modelo no puede saber cuál va en cada lugar. Mira el ejemplo: predice `[1, 1, 1, 1, 1, 1, 1, 1]`, ¡el dígito que se repite en la entrada!

¿Por qué no 10 %? Porque hace trucos: si un dígito aparece varias veces en la entrada, apostar por él acierta más que el azar, y cada posición también conoce **su propio** dígito, que a veces coincide con el de su espejo. Pero invertir de verdad es **imposible** sin saber el orden.
</details>

---
## 6. 📖 Hiperparámetros del Transformer

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `d_model` | Tamaño del vector de cada token | Más capacidad, más memoria y cálculo | Más barato; si es muy chico, no cabe la información | 64–512 (pequeños) · 768 (GPT-2) |
| `n_cabezas` | Número de búsquedas en paralelo | Más tipos de relaciones, cada una con vectores más chicos | Menos variedad de relaciones | `d_model / 64` |
| `n_capas` | Número de bloques apilados | Razonamientos de más pasos; más difícil de entrenar | Más rápido, menos expresivo | 2–6 (pequeños) · 12 (GPT-2) |
| `d_ff` (tamaño interno del MLP) | Capacidad de "reflexión" por token | Más capacidad | Menos parámetros | 4 × `d_model` |
| Longitud de contexto | Cuántos tokens ve a la vez | Más contexto, costo n² | Más barato | Lo que necesite la tarea |
| `dropout` | Regularización (Lección 9.8) | Menos sobreajuste, aprende más lento | Aprende rápido, puede sobreajustar | 0.0–0.1 |
| Tasa de aprendizaje + *warmup* | Tamaño del paso (Lección 9.6) | Inestable si es muy alta | Lento | 3e-4 con *warmup*; 1e-3–3e-3 en modelos diminutos |

### ✏️ Ejercicio 9 — ¿Qué tan pequeño puede ser?
El modelo de arriba usa `d_model = 32`. La celda entrena el mismo modelo con vectores de **32, 8 y 4** casillas (y cabezas acordes). ¿Con cuál crees que empieza a fallar? ✏️ Luego prueba tus propias combinaciones (recuerda: `d_model` tiene que ser divisible entre `n_cabezas`).

✍️ *Mi predicción:*

In [ ]:
CONFIGURACIONES = [dict(d_model=32, n_cabezas=4), dict(d_model=8, n_cabezas=2), dict(d_model=4, n_cabezas=1)]   # ✏️

for cfg in CONFIGURACIONES:
    m = entrenar("invertir", pasos=300, **cfg)
    dig, sec = evaluar(m)
    print(f"{str(cfg):<35} parámetros: {sum(p.numel() for p in m.parameters()):>6,} · dígitos {dig:.0%} · secuencias {sec:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- `d_model = 32` → ≈100 %.
- `d_model = 8` → ¡todavía ≈100 %, con solo ~2 mil parámetros! Para una tarea tan simple sobra capacidad.
- `d_model = 4` → se derrumba (≈17 % de dígitos, peor incluso que sin posiciones). En 4 casillas tiene que guardar **a la vez** qué dígito es y en qué posición está: no cabe.

Es la misma historia de siempre: un modelo demasiado pequeño se queda en **subajuste** (Lección 4.4). En la Lección 12.5 verás cómo crece la calidad con el tamaño.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| `AssertionError: embed_dim must be divisible by num_heads` | `d_model` no es divisible entre `n_cabezas` | Elegir, p. ej., 64/4, 128/8, 768/12 |
| El modelo trata "el perro mordió al hombre" igual que "el hombre mordió al perro" | Falta la codificación posicional | Sumar `nn.Embedding(largo, d_model)` o la sinusoidal |
| Error al usar frases más largas que en el entrenamiento | La tabla de posiciones aprendida solo tiene `largo` filas | Entrenar con el contexto máximo que usarás (o usar posiciones sinusoidales/relativas) |
| La pérdida explota o da `nan` al apilar muchas capas | Faltan atajos residuales o normalización, o la tasa de aprendizaje es muy alta | Revisar `x + …`, LayerNorm, bajar el *lr*, usar *warmup* |
| Una cabeza "no hace nada" | Normal: hay cabezas redundantes | Se pueden podar (Proyecto del módulo) |

### 🔮 Ejercicio 10 — 32 casillas y 5 cabezas
¿Qué pasa si pides 5 cabezas con `d_model = 32`?

✍️ *Mi predicción:*

In [ ]:
try:
    nn.MultiheadAttention(embed_dim=32, num_heads=5)      # ✏️ prueba 32 y 8
    print("✅ Funciona")
except AssertionError as error:
    print("❌ Error:", error)

<details><summary><b>👀 Ver respuesta</b></summary>

Error: **`embed_dim must be divisible by num_heads`**. Cada cabeza recibe un pedazo de `d_model / n_cabezas` casillas; 32 / 5 = 6.4 no es un número entero. Con 8 cabezas (32 / 8 = 4) funciona.
</details>

---
## 8. 📋 Ficha del modelo: Transformer

| | Transformer |
|---|---|
| **Tipo** | Red neuronal para secuencias (texto, audio, imágenes en parches, proteínas…) |
| **Intuición** | Capas que alternan una **reunión** (atención: las palabras intercambian información) y una **reflexión** (MLP: cada palabra procesa por su cuenta) |
| **Piezas** | *Embedding* + código de posición → N bloques (atención multi-cabeza + MLP, con residuales y LayerNorm) → capa de salida |
| **Cómo aprende** | Descenso de gradiente sobre una pérdida (p. ej. entropía cruzada de la siguiente palabra) |
| **Hiperparámetros clave** | `d_model`, `n_cabezas`, `n_capas`, longitud de contexto, tasa de aprendizaje |
| **✅ Fortalezas** | Conecta palabras lejanas en un paso · Entrenamiento muy paralelo (GPUs) · Escala muy bien (Lección 12.5) · Una sola arquitectura para muchos tipos de datos |
| **❌ Debilidades** | Costo n² con la longitud · Necesita muchos datos · No conoce el orden sin codificación posicional · Caro de entrenar a gran escala |
| **Úsalo cuando…** | Tienes secuencias y suficientes datos (o un modelo preentrenado que ajustar) |
| **Evítalo cuando…** | Tienes pocos datos tabulares (usa boosting, M6) o secuencias enormes sin recursos |

## 9. 🧠 Resumen en 5 puntos
1. **Multi-cabeza:** varias atenciones en paralelo, cada una con sus matrices; buscan relaciones distintas y **no** aumentan los parámetros.
2. La auto-atención es **ciega al orden**: sin posiciones, "el perro mordió al hombre" = "el hombre mordió al perro".
3. La **codificación posicional** (sinusoidal o aprendida) suma a cada token un código de su posición.
4. Un **bloque Transformer** = atención (reunión) + MLP (reflexión), con atajos residuales y normalización; se apilan bloques idénticos.
5. Un mini-Transformer de ~26 mil parámetros aprende a invertir secuencias en segundos, y sus cabezas **se reparten el trabajo solas**.

---
## 10. 🏋️ Ejercicios finales

### 10.1 Repaso rápido (opción múltiple)

**1.** Pasar de 1 a 8 cabezas (mismo `d_model`) hace que la atención tenga…
- a) 8 veces más parámetros · b) los mismos parámetros · c) 8 veces menos parámetros · d) ningún parámetro

**2.** ¿Qué pieza le dice al Transformer en qué lugar de la frase está cada palabra?
- a) La normalización · b) El MLP · c) La codificación posicional · d) La máscara causal

**3.** En un bloque Transformer, la parte donde las palabras **intercambian** información es…
- a) El MLP · b) La atención · c) La normalización · d) El atajo residual

**4.** ¿Para qué sirve el atajo residual (`x + …`)?
- a) Para que el modelo conozca el orden · b) Para que cada capa solo sume una corrección y se entrenen bien muchas capas · c) Para reducir parámetros · d) Para tapar el futuro

**5.** Un modelo usa `d_model = 768` y 12 cabezas. ¿De qué tamaño son los vectores de cada cabeza?
- a) 12 · b) 64 · c) 768 · d) 9,216

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Las cabezas reparten el mismo `d_model` (Ejercicio 2).
2. **c)** Sin ella, la atención ve una bolsa de palabras.
3. **b)** La atención es la "reunión"; el MLP es la "reflexión" individual.
4. **b)** Es la idea de ResNet (Lección 10.4).
5. **b)** 768 / 12 = 64.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Un compañero dice: "Para que el modelo distinga el orden, mejor le agrego a cada palabra un número: 1, 2, 3… hasta 1,000, en una casilla extra". ¿Qué problema le ves frente a la codificación sinusoidal o aprendida? (Pista: piensa en el tamaño de los números y en la Lección 2.3, escalado.)

**B. Caso 💼** Una empresa quiere un Transformer para leer **conjuntos** de síntomas (fiebre, tos, dolor de cabeza…) y sugerir un diagnóstico. El orden en que el médico escribe los síntomas **no** importa. ¿Debería usar codificación posicional? ¿Por qué?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Números de 1 a 1,000 tienen escalas muy distintas a las del resto de casillas (que rondan entre −1 y 1): esa casilla dominaría los productos punto, como una variable sin escalar domina en KNN. Además, la posición 999 nunca vista sería un número "fuera de rango". Los códigos sinusoidales o aprendidos son vectores **de tamaño acotado** y de muchas casillas, fáciles de comparar entre sí.

**B.** **No.** Si el orden no importa, que el modelo sea "ciego al orden" es una **ventaja**: dará el mismo diagnóstico sin importar cómo se escribió la lista. Agregar posiciones le haría aprender diferencias falsas ("tos antes que fiebre"). Un Transformer sin codificación posicional es un modelo natural para **conjuntos**.
</details>

### 10.3 🧩 Práctica integradora: un Transformer que ordena

Nueva tarea: recibir 8 dígitos y devolverlos **ordenados** de menor a mayor.

```
entrada:  3 1 4 1 5 9 2 6
salida :  1 1 2 3 4 5 6 9
```

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 🔮 — Antes de entrenar.** Para **ordenar**, el orden de la **entrada** no importa (los mismos dígitos en otro orden dan la misma salida). ¿Crees que el modelo necesitará codificación posicional? Piénsalo bien: ¿qué necesita saber cada posición de la **salida**?

✍️ *Mi predicción:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Sí la necesita.** La entrada puede verse como un conjunto, pero cada posición de **salida** tiene que saber **qué lugar ocupa**: la posición 0 debe escribir el más pequeño, la 7 el más grande. Sin posiciones, las 8 salidas son indistinguibles y escribirían lo mismo (lo comprobarás en el Paso 4).
</details>

**Paso 2 ▶️ — Entrena.** Usamos la misma función `entrenar`, ahora con la tarea `"ordenar"`.

In [ ]:
modelo_ord = entrenar("ordenar", pasos=300)
dig_ord, sec_ord = evaluar(modelo_ord, "ordenar")
print(f"Ordenar → dígitos: {dig_ord:.1%} · secuencias perfectas: {sec_ord:.1%}")
print("Entrada :", x_ej[0].tolist())
print("Predicho:", modelo_ord(x_ej)[0].argmax(-1)[0].tolist())

✋ ¿Qué tal lo hizo comparado con invertir?

<details><summary><b>👀 Ver respuesta</b></summary>

También ≈**100 %** de dígitos y ≈98 % de secuencias perfectas en 300 pasos. Ordenar parece más difícil que invertir, pero para un Transformer es natural: cada salida puede **mirar a todos los dígitos a la vez** y contar cuántos son menores.
</details>

**Paso 3 ▶️ — Mira la atención.** Usamos una entrada con dígitos fáciles de seguir. Mira la **última capa**: en cada fila (posición de salida), ¿a qué columnas (posiciones de entrada) mira?

In [ ]:
x_ord = torch.tensor([[7, 0, 5, 2, 9, 4, 1, 8]], device=device)
salida_ord, pesos_ord = modelo_ord(x_ord)
print("Entrada :", x_ord[0].tolist())
print("Predicho:", salida_ord.argmax(-1)[0].tolist())

pesos_prom = pesos_ord[1][0].mean(0).detach().cpu().numpy()      # capa 1, promedio de las 4 cabezas
etiquetas = [f"{p}:{d}" for p, d in enumerate(x_ord[0].tolist())]  # "posición:dígito"
mapa_calor(pesos_prom, range(LARGO), etiquetas, "Capa 1 (promedio de cabezas)")
plt.xlabel("posición:dígito de la entrada"); plt.ylabel("posición de la salida"); plt.show()

✍️ *¿Qué patrón ves?*

<details><summary><b>👀 Ver respuesta</b></summary>

La salida es `[0, 1, 2, 4, 5, 7, 8, 9]`: correcta. En el mapa, las casillas más oscuras de cada fila suelen caer sobre el dígito que esa posición tiene que escribir **o sobre sus vecinos en valor**: la fila 0 mira al 0 y al 1 (columnas "1:0" y "6:1"), la fila 1 al 1 ("6:1"), la fila 6 al 8 ("7:8") y la fila 7 se reparte entre el 8 y el 9. Las filas intermedias reparten su atención entre varios candidatos: la atención **busca por valor**, no por posición de entrada. Con la idea de la biblioteca: la salida 0 consulta "el más pequeño" y los dígitos anuncian "soy un 0, soy un 7…".
</details>

**Paso 4 🔮 — Sin posiciones.** Repite el entrenamiento con `con_posicion=False`. Predice el resultado y lo que verás en "Predicho".

In [ ]:
modelo_ord_sin = entrenar("ordenar", pasos=300, con_posicion=False)
d, s = evaluar(modelo_ord_sin, "ordenar")
print(f"Ordenar SIN posición → dígitos: {d:.1%} · secuencias perfectas: {s:.1%}")
print("Predicho:", modelo_ord_sin(x_ord)[0].argmax(-1)[0].tolist())

<details><summary><b>👀 Ver respuesta</b></summary>

≈**30 %** de dígitos y 0 % de secuencias, como en el Ejercicio 8. Y en "Predicho" las salidas no tienen orden y se repiten (el 9 aparece cuatro veces): sin saber su lugar, cada posición de salida solo puede apostar por un dígito "típico" de la entrada. Confirmas el Paso 1: **las posiciones de salida necesitan saber dónde están**.
</details>

**Paso 5 ✋ — Para tu cuaderno.** Resume en 3 frases lo que aprendiste de las dos tareas (invertir y ordenar): qué papel juegan la atención, la posición y las cabezas.

<details><summary><b>👀 Ver una respuesta modelo</b></summary>

1. La **atención** permite que cada posición de salida reúna información de cualquier posición de entrada en un solo paso: para invertir mira a su espejo; para ordenar busca dígitos por su valor.
2. La **posición** es imprescindible siempre que la respuesta dependa del lugar: sin ella, ambos modelos se quedan en ≈30 %.
3. Las **cabezas** se reparten el trabajo de forma espontánea; mirar sus mapas ayuda a entender (y a depurar) qué aprendió el modelo, como harás en el Proyecto del módulo.
</details>

---
## 📚 Para profundizar
- 📄 **Artículo:** Vaswani y colegas (2017), *Attention Is All You Need* (secciones 3.2.2, multi-cabeza, y 3.5, codificación posicional).
- 📺 **Video:** Andrej Karpathy, *Let's build GPT: from scratch, in code, spelled out* (inglés): construye un Transformer de caracteres paso a paso.
- 📘 **Blog:** Harvard NLP, *The Annotated Transformer*: el artículo original con su código línea por línea.
- 🎮 **Interactivo:** *Transformer Explainer* (poloclub.github.io/transformer-explainer).

---
⬅️ **Anterior:** [12.2 Auto-atención: consultas, claves y valores (la biblioteca)](12.2-auto-atencion-qkv.ipynb) · ➡️ **Siguiente:** [12.4 Encoder (BERT) vs. Decoder (GPT) vs. Encoder-Decoder (T5)](12.4-encoder-decoder.ipynb)